# PyDI Data Integration Workflow: Videogames

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with vidoegame datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
  - [Step 1: Load Target Schema and Normalization Spec](#step-1-load-target-schema-and-normalization-spec)
  - [Step 2: Load Source Datasets](#step-2-load-source-datasets)
  - [Step 3: LLM-Based Schema Matching](#step-3-llm-based-schema-matching)
  - [Step 4: Schema Matching Evaluation](#step-4-evaluate-schema-matching-against-gold-mapping)
  - [Step 5: Translate and Normalize](#step-5-translate-and-normalize)
- [Part 2: Data Profiling](#part-2-data-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

### Datasets

- **DBpedia**: 65,000 records
- **Metacritic**: 20,494 records
- **Global Sales Ranking**: 7,877 records

## Part 1: Schema Matching and Value Normalization

In [ ]:
import time
start_time = time.time()

from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaMappingEvaluator, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

## Step 1: Load Target Schema and Normalization Spec

In [ ]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")

# Set genres column manually to list type
spec.set_column("genres", output_type="list")

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

## Step 2: Load Source Datasets

In [ ]:
from PyDI.io import load_xml, load_csv, load_json
dbpedia = load_csv(INPUT_DIR / "data" / "dbpedia.csv")
dbpedia.attrs["dataset_name"] = "dbpedia"
dbpedia.head()

In [ ]:
metacritic = load_csv(INPUT_DIR / "data" / "metacritic.csv")
metacritic.attrs["dataset_name"] = "metacritic"
metacritic.head()

In [ ]:
sales = load_csv(INPUT_DIR / "data" / "sales.csv")
sales.attrs["dataset_name"] = "sales"
sales.head()

## Step 3: LLM-Based Schema Matching

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match dbpedia dataset
dbpedia_mapping = matcher.match(dbpedia, df_target)

dbpedia_mapping

In [ ]:
metacritic_mapping = matcher.match(metacritic, df_target)
metacritic_mapping

In [ ]:
sales_mapping = matcher.match(sales, df_target)
sales_mapping

## Step 4: Evaluate Schema Matching Against Gold Mapping


In [ ]:
# Load the manually curated schema-matching gold standard
with open(INPUT_DIR / "schemamatching" / "sm_mapping_gold.json") as f:
    schema_mapping_gold = json.load(f)

schema_mapping_gold_df = pd.DataFrame(schema_mapping_gold["mappings"])
schema_mapping_predictions = pd.concat(
    [dbpedia_mapping, metacritic_mapping, sales_mapping],
    ignore_index=True,
)

schema_matching_metrics = SchemaMappingEvaluator.evaluate(
    schema_mapping_predictions,
    schema_mapping_gold_df,
    complete=True,
)

schema_matching_summary = pd.DataFrame([
    {
        **schema_matching_metrics,
        "predicted_mappings": len(schema_mapping_predictions),
        "gold_mappings": len(schema_mapping_gold_df),
    }
])

per_source_schema_matching = pd.DataFrame([
    {
        "source_dataset": source_dataset,
        **SchemaMappingEvaluator.evaluate(
            schema_mapping_predictions[
                schema_mapping_predictions["source_dataset"] == source_dataset
            ],
            schema_mapping_gold_df[
                schema_mapping_gold_df["source_dataset"] == source_dataset
            ],
            complete=True,
        ),
    }
    for source_dataset in sorted(schema_mapping_gold_df["source_dataset"].unique())
])

display(schema_matching_summary)
per_source_schema_matching


## Step 5: Translate and Normalize


In [ ]:
# Unify platform names across datasets
platform_groups = {
    "Nintendo Entertainment System": ["NES"],
    "Super Nintendo": ["SNES", "Super Nintendo Entertainment System"],
    "Nintendo 64": ["N64"],
    "GameCube": ["Nintendo GameCube", "GC"],
    "Wii": ["Nintendo Wii"],
    "Game Boy Color": ["GBC"],
    "Game Boy Advance": ["GBA"],
    "Game Boy": ["GB"],
    "DS": ["Nintendo DS"],
    "3DS": ["Nintendo 3DS"],
    "Switch": ["Nintendo Switch"],

    "Playstation": [
        "Playstation (console)", "PlayStation (console)",
        "Playstation 1", "PlayStation 1",
        "PS1", "PSX", "PS"
    ],

    "PS2": ["Playstation 2", "PlayStation 2"],
    "PS3": ["Playstation 3", "PlayStation 3"],
    "PS4": ["Playstation 4", "PlayStation 4"],
    "Playstation Portable": ["PSP"],
    "Playstation Vita": ["PSV", "PS Vita"],
    "Playstation VR": ["PSVR", "PS VR"],

    "Xbox": ["XB", "Xbox (console)"],
    "Xbox One": ["XOne"],
    "Xbox 360": ["X360"],

    "PC": ["Microsoft Windows", "Windows"],
}

platform_map = {}
for canonical, variants in platform_groups.items():
    for alias in variants:
        platform_map[alias.lower()] = canonical

def normalize_platform(series):
    def _norm(x):
        # Leave missing or non-string values as they are
        if not isinstance(x, str):
            return x
        key = x.strip().lower()
        return platform_map.get(key, x.strip())
    
    return series.apply(_norm)

dbpedia["system"] = normalize_platform(dbpedia["system"])
metacritic["console"] = normalize_platform(metacritic["console"])
sales["hw"] = normalize_platform(sales["hw"])

In [ ]:
# Create id columns based on index (starting with 1)
sales["id"] = sales.index + 1
metacritic["id"] = metacritic.index + 1
dbpedia["id"] = dbpedia.index + 1

# Make id entries more explicit
dbpedia["id"] = dbpedia["id"].apply(lambda x: f"dbpedia_{x}")
metacritic["id"] = metacritic["id"].apply(lambda x: f"metacritic_{x}")
sales["id"] = sales["id"].apply(lambda x: f"sales_{x}")

In [ ]:
# Clean up noisy dbpedia dataset

# Remove suffix (" (video game)") from franchise
dbpedia_cleaned = dbpedia.copy()
dbpedia_cleaned["franchise"] = dbpedia_cleaned["franchise"].str.replace(r" \(video game\)$", "", regex=True)


In [ ]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("releaseYear", output_type="datetime")
dbpedia_normalized = translator.translate(
    dbpedia_cleaned, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse special date format (e.g., Oct 26, 2018)
spec.set_column("releaseYear", output_type="datetime", date_format="%b %d, %Y")

metacritic_normalized = translator.translate(
    metacritic, metacritic_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse year-only values (e.g., "1929", "2010")
spec.set_column("releaseYear", output_type="datetime", date_format="%Y")

sales_normalized = translator.translate(
    sales, sales_mapping,
    normalize=spec, on_failure="keep"
)

In [ ]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head()

In [ ]:
metacritic_cols = [c for c in target_columns if c in metacritic_normalized.columns]
metacritic_normalized[metacritic_cols].head()

In [ ]:
sales_cols = [c for c in target_columns if c in sales_normalized.columns]
sales_normalized[sales_cols].head()

In [ ]:
# convert genre strings to lists
normalized_sets = [dbpedia_normalized, metacritic_normalized, sales_normalized]
for s in normalized_sets:
    s["genres"] = s["genres"].apply(lambda x: [g.strip() for g in x.split(",")] if isinstance(x, str) else x)


In [ ]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
metacritic = metacritic_normalized[metacritic_cols].copy()
sales = sales_normalized[sales_cols].copy()

In [ ]:
# --- MaDI-Bench normalization output: the normalized source tables, one CSV per source
# (record id + target-schema columns), the input of madi_bench.evaluation.score_normalization,
# which scores them against input/normalization/<split>.csv.
import json as _json
from pathlib import Path as _Path
_norm_dir = OUTPUT_DIR / "normalization"
_norm_dir.mkdir(parents=True, exist_ok=True)
_target = list(_json.loads((_Path(NOTEBOOK_DIR) / "input" / "schemamatching" / "target_schema.json").read_text())["properties"])
for _name, _df in {"dbpedia": dbpedia_normalized, "metacritic": metacritic_normalized, "sales": sales_normalized}.items():
    _id_col = "id" if "id" in _df.columns else next((c for c in _df.columns if c.endswith("_id")), _df.columns[0])
    _cols = [_id_col] + [c for c in _df.columns if c in _target and c != "id" and c != _id_col]
    _df[_cols].rename(columns={_id_col: "id"}).to_csv(_norm_dir / f"{_name}.csv", index=False)
    print(f"normalized table {_name}: {len(_df)} rows, {len(_cols)} columns -> {_norm_dir / (_name + '.csv')}")


## Part 2: Data Profiling

In [ ]:
# Display basic information
datasets = [dbpedia, metacritic, sales]
names = ["DBpedia", "Metacritic", "Sales"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

In [ ]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

### Attribute Coverage Analysis

In [ ]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print(" Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

## Part 3: Entity Matching

### Step 1: Blocking

In [ ]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [ ]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re
import pandas as pd

TITLE_TOKEN_STOPWORDS = {
    "the", "of", "and", "a", "an", "in", "to", "for", "on", "with", "at", "by", "from",
    "edition", "game", "video", "ii", "iii", "iv"
}

def title_tokens(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name).lower())
    return [
        token for token in tokens
        if len(token) >= 3 and token not in TITLE_TOKEN_STOPWORDS
    ]

def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

platform_aliases = {
    "playstation 4": "ps4", "ps4": "ps4",
    "playstation 3": "ps3", "ps3": "ps3",
    "playstation 2": "ps2", "ps2": "ps2",
    "playstation vita": "ps vita", "ps vita": "ps vita", "psv": "ps vita",
    "playstation portable": "psp", "psp": "psp",
    "xbox one": "xbox one", "xone": "xbox one",
    "xbox 360": "xbox 360", "x360": "xbox 360",
    "xbox": "xbox",
    "nintendo switch": "switch", "switch": "switch",
    "wii": "wii", "wii u": "wii u",
    "gamecube": "gamecube", "nintendo gamecube": "gamecube",
    "nintendo ds": "ds", "ds": "ds",
    "nintendo 3ds": "3ds", "3ds": "3ds",
    "game boy advance": "gba", "gba": "gba",
    "game boy color": "gbc", "gbc": "gbc",
    "pc": "pc", "microsoft windows": "pc", "windows": "pc",
    "macintosh": "mac", "mac os x": "mac",
    "ios": "ios", "android": "android",
    "arcade video game": "arcade", "arcade": "arcade",
}

def normalize_platform_value(value):
    key = str(value).strip().lower()
    return platform_aliases.get(key, key)

def release_year_value(value):
    match = re.search(r"\d{4}", str(value))
    return match.group(0) if match else ""

class UnionTitleTokenBlocker:
    """Union of shared title-token blocks constrained by platform or release year."""

    def __init__(self, df_left, df_right, *, id_column, name_column, platform_column, year_column, batch_size=100_000):
        self.df_left = df_left
        self.df_right = df_right
        self.id_column = id_column
        self.name_column = name_column
        self.platform_column = platform_column
        self.year_column = year_column
        self.batch_size = int(batch_size)
        self.rules = (
            ("title_token_platform", "platform_block_key"),
            ("title_token_year", "release_year_block_key"),
        )
        self._left_blocks = self._build_blocks(df_left)
        self._right_blocks = self._build_blocks(df_right)

    def _build_blocks(self, df):
        blocks = {rule_name: {} for rule_name, _ in self.rules}
        for row in df[[self.id_column, self.name_column, self.platform_column, self.year_column]].itertuples(index=False):
            record_id = getattr(row, self.id_column)
            name = getattr(row, self.name_column)
            platform = normalize_platform_value(getattr(row, self.platform_column))
            release_year = release_year_value(getattr(row, self.year_column))
            for token in title_tokens(name):
                if platform:
                    blocks["title_token_platform"].setdefault((token, platform), []).append(record_id)
                if release_year:
                    blocks["title_token_year"].setdefault((token, release_year), []).append(record_id)
        return blocks

    def __iter__(self):
        seen_pairs = set()
        batch = []
        for rule_name, _ in self.rules:
            left_blocks = self._left_blocks[rule_name]
            right_blocks = self._right_blocks[rule_name]
            for block_key in left_blocks.keys() & right_blocks.keys():
                for left_id in left_blocks[block_key]:
                    for right_id in right_blocks[block_key]:
                        pair = (left_id, right_id)
                        if pair in seen_pairs:
                            continue
                        seen_pairs.add(pair)
                        batch.append((left_id, right_id, f"{rule_name}:{block_key[0]}:{block_key[1]}"))
                        if len(batch) >= self.batch_size:
                            yield pd.DataFrame(batch, columns=["id1", "id2", "block_key"])
                            batch = []
        if batch:
            yield pd.DataFrame(batch, columns=["id1", "id2", "block_key"])

    def materialize(self):
        frames = [batch for batch in self if not batch.empty]
        if frames:
            return pd.concat(frames, ignore_index=True)
        return pd.DataFrame(columns=["id1", "id2", "block_key"])

    def estimate_pairs(self):
        return sum(len(batch) for batch in self)


dbpedia['name_longest_token'] = dbpedia['name'].apply(get_longest_token)
metacritic['name_longest_token'] = metacritic['name'].apply(get_longest_token)
sales['name_longest_token'] = sales['name'].apply(get_longest_token)

standard_blocker_m2d = UnionTitleTokenBlocker(
    dbpedia,
    metacritic,
    id_column='id',
    name_column='name',
    platform_column='platform',
    year_column='releaseYear',
    batch_size=100_000,
)


### Step 2: Evaluate Blocking Against Ground Truth

In [ ]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_metacritic_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on the union title-token blocker
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

In [ ]:
# def get_first_token(name):
#     return re.split(r"[^A-Za-z0-9_']+", str(name))[0]
# metacritic['name_first_token'] = metacritic['name'].apply(get_first_token)
# sales['name_first_token'] = sales['name'].apply(get_first_token)

standard_blocker_m2s = StandardBlocker(
    dbpedia, sales,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_sales_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2s,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

Now let's evaluate which blocking method we want to use for each dataset combination:

### Step 3: Entity Matching with Comparators

In [ ]:
from PyDI.entitymatching import StringComparator, DateComparator
import difflib

# Create comparators for different attributes
# DBpedia-Metacritic uses a stricter title/platform rule because the high-recall
# blocker intentionally admits many near-title candidates.
def normalize_match_title(value):
    value = str(value).lower()
    value = re.sub(r"\([^)]*video game[^)]*\)", " ", value)
    value = re.sub(
        r"\b(video game|game|hd|remaster(?:ed)?|definitive edition|special edition|complete edition|goty edition)\b",
        " ",
        value,
    )
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

MATCH_TITLE_STOPWORDS = {
    "the", "of", "and", "a", "an", "in", "to", "for", "on", "with", "at", "by", "from",
    "edition", "game", "video"
}

def match_title_tokens(value):
    return {
        token for token in normalize_match_title(value).split()
        if len(token) >= 2 and token not in MATCH_TITLE_STOPWORDS
    }

def name_sequence_similarity(record1, record2):
    return difflib.SequenceMatcher(
        None,
        normalize_match_title(record1["name"]),
        normalize_match_title(record2["name"]),
    ).ratio()

def name_token_overlap_similarity(record1, record2):
    tokens1 = match_title_tokens(record1["name"])
    tokens2 = match_title_tokens(record2["name"])
    if not tokens1 or not tokens2:
        return 0.0
    return len(tokens1 & tokens2) / min(len(tokens1), len(tokens2))

def platform_exact_similarity(record1, record2):
    platform1 = normalize_platform_value(record1["platform"])
    platform2 = normalize_platform_value(record2["platform"])
    return 1.0 if platform1 and platform2 and platform1 == platform2 else 0.0

comparators_m2d = [
    name_sequence_similarity,
    name_token_overlap_similarity,
    platform_exact_similarity,
]

comparators_m2s = [
    StringComparator(
        column='name',
        similarity_function='jaccard',
        preprocess=str.lower
    ),
        # Platform similarity - supporting evidence
    StringComparator(
        column='platform',
        similarity_function='jaccard',
    ),
    DateComparator(
        column='releaseYear',
        max_days_difference=360  # Allow almost 1 year difference
    )
]


Next, we setup the matcher and run the matching with our chosen best blocking method:

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=dbpedia,
    df_right=metacritic, 
    candidates=standard_blocker_m2d, # union title-token blocker: token+platform OR token+releaseYear
    comparators=comparators_m2d,
    weights=[0.65, 0.25, 0.10], # title sequence, title token overlap, platform
    threshold=0.98, # tuned for the high-recall DBpedia-Metacritic blocker
    id_column='id'
)

In [ ]:
correspondences_m2s = matcher.match(
    df_left=dbpedia,
    df_right=sales, 
    candidates=standard_blocker_m2s,
    comparators=comparators_m2s,
    weights=[0.6, 0.3, 0.1], # name, platform, releaseYear
    threshold=0.8,
    id_column='id'
)

### Step 4: Evaluate Matching Against Ground Truth

In [ ]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_metacritic_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

In [ ]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

In [ ]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [ ]:
from PyDI.entitymatching import MaximumBipartiteMatching
     
clusterer = MaximumBipartiteMatching()
correspondences_m2d_post = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d_post,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d_post,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

In [ ]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_sales_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

clusterer = MaximumBipartiteMatching()
correspondences_m2s_post = clusterer.cluster(correspondences_m2s)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s_post,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s_post,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)


## Part 4: Data Fusion

In [ ]:
# We are only interested in year of release -> normalize all dates to YYYY-01-01.
# Coerce first so this works whether releaseYear is already datetime or still a string/object column.
def normalize_to_year_start(series):
    dates = pd.to_datetime(series, errors="coerce")
    return dates.dt.to_period("Y").dt.to_timestamp()

for dataset in [dbpedia, metacritic, sales]:
    dataset["releaseYear"] = normalize_to_year_start(dataset["releaseYear"])


In [ ]:
metacritic["metacritic_id"] = metacritic["id"]

# Assign trust scores to datasets
metacritic.attrs["trust_score"] = 3
sales.attrs["trust_score"] = 2
dbpedia.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2s], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

## Step 1: Define Fusion Strategy 

In [ ]:
# Identify which attributes are worth fusing:
# attributes that appear in more than one source AND in the fusion groundtruth.
import xml.etree.ElementTree as ET
from collections import defaultdict

sources = {"metacritic": metacritic, "dbpedia": dbpedia, "sales": sales}

# Target schema attributes (skip 'id' since it is the join key, not fused)
target_attrs = [a for a in target_schema["properties"].keys() if a != "id"]

# Groundtruth attributes from the XML directly, so nested elements like
# <genres><genre>...</genre></genres> count as the parent tag "genres".
gt_tree = ET.parse(INPUT_DIR / "fusion" / "validation_set.xml")
gt_attrs = {child.tag for vg in gt_tree.getroot().findall("videogame") for child in vg}

# For each target attribute, which sources actually contain it?
attr_sources = defaultdict(list)
for src_name, df in sources.items():
    for attr in target_attrs:
        if attr in df.columns:
            attr_sources[attr].append(src_name)

rows = []
for attr in target_attrs:
    srcs = attr_sources.get(attr, [])
    rows.append({
        "attribute": attr,
        "num_sources": len(srcs),
        "sources": ", ".join(srcs) if srcs else "—",
        "in_groundtruth": attr in gt_attrs,
        "fuse?": len(srcs) > 1 and attr in gt_attrs,
    })
overview = pd.DataFrame(rows).sort_values(["fuse?", "num_sources"], ascending=[False, False])
display(overview)

fusion_attributes = sorted(overview.loc[overview["fuse?"], "attribute"].tolist())
print("Attributes to fuse:", fusion_attributes)


In [ ]:
from PyDI.fusion import DataFusionStrategy, prefer_higher_trust, voting, average, union

strategy = DataFusionStrategy('game_fusion_strategy')
['id', 'ESRB', 'criticScore', 'developer', 'genres', 'name', 'platform', 'releaseYear', 'userScore']
strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('platform', voting)
strategy.add_attribute_fuser('developer', voting)
strategy.add_attribute_fuser('releaseYear', voting, trust_key="trust_score")
strategy.add_attribute_fuser('ESRB', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('criticScore', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('userScore', average)
strategy.add_attribute_fuser('genres', union)

## Step 2: Run Fusion

In [ ]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[metacritic, dbpedia, sales],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

## Step 3: Evaluate Data Fusion

In [ ]:
from PyDI.fusion import tokenized_match, year_only_match, boolean_match, numeric_tolerance_match, exact_match, intersection
['genres', 'name', 'platform', 'releaseYear', 'userScore']
strategy.add_evaluation_function("name", exact_match)
strategy.add_evaluation_function("platform", exact_match)
strategy.add_evaluation_function("developer", exact_match)
strategy.add_evaluation_function("releaseYear", year_only_match)
strategy.add_evaluation_function("ESRB", exact_match)
strategy.add_evaluation_function("criticScore", numeric_tolerance_match, tolerance=2)
strategy.add_evaluation_function("userScore", numeric_tolerance_match, tolerance=0.2)
strategy.add_evaluation_function("genres", intersection)

In [ ]:
# --- MaDI-Bench fusion evaluation: the public evaluation package scores the
# fused table on the validation and the test split -- strict comparators on the v2 gold set,
# all-gold accuracy (a gold record the pipeline did not produce counts as wrong) with its coverage.
# This is the scorer the benchmark's reference numbers were produced with; the notebook's own
# comparator registrations above are not used for scoring.
import json as _json, os as _os, sys as _sys
from pathlib import Path as _Path
try:
    from madi_bench.evaluation import score_fusion as _score_fusion
except ImportError:
    _sys.path.insert(0, _os.environ.get("MADI_BENCH_ROOT") or str(_Path(NOTEBOOK_DIR).resolve().parents[2]))
    from madi_bench.evaluation import score_fusion as _score_fusion
_fused_out = fused.copy()
# the notebook's own column names for target-schema attributes (the same aliases the benchmark's
# rescoring applies, reproduction/scoring/rescore.py): scoring is by target-schema name
for _src, _dst in ():
    if _src in _fused_out.columns and _dst not in _fused_out.columns:
        _fused_out = _fused_out.rename(columns={_src: _dst})
_fusion_dir = OUTPUT_DIR / "data_fusion"
_fusion_dir.mkdir(parents=True, exist_ok=True)
_fused_out.to_csv(_fusion_dir / "fused.csv", index=False)
_scores = {}
for _split in ("validation", "test"):
    _r = _score_fusion(NOTEBOOK_DIR, _fused_out, split=_split)
    _scores[_split] = _r
    print(f"{_split}: all-gold accuracy {_r['overall_accuracy_all_gold']:.4f} "
          f"(evaluated {_r['overall_accuracy_evaluated']:.4f} on {_r['n_gold_evaluated']}/{_r['n_gold']} gold records, "
          f"attribute cell coverage {_r['attribute_cell_coverage']:.2f})")
    for _a, _v in sorted(_r["per_attribute_accuracy_all_gold"].items()):
        print(f"    {_a:28s} {_v:.4f}")
(_fusion_dir / "fusion_scores_v2.json").write_text(_json.dumps(_scores, indent=1, default=str))


In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal execution time: {elapsed_time:.2f} seconds")